# Step 4: Exploratory data analysis of MovieLens `ml-latest-small`

Purpose: find out what the data looks like *before* choosing any model or evaluation split.
This notebook shows aggregate statistics and charts only (no raw rows). It uses the loaders from
`pmrs.data`, which check the files and drop nothing.

Data: MovieLens `ml-latest-small` (GroupLens Research, University of Minnesota). Citation:
F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and Context.*
ACM TiiS 5, 4, Article 19. https://doi.org/10.1145/2827872. Non-commercial research use only.

This is a development dataset (per its README), so the project compares methods with it; the
numbers are not benchmark results. **This notebook does not choose evaluation cutoffs; that is Step 5.**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from pmrs.data import genres_multi_hot, load_movielens

pio.renderers.default = "plotly_mimetype"  # small JSON outputs, no embedded plotly.js
pd.set_option("display.max_rows", 25)
pd.set_option("display.width", 100)

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
data = load_movielens(root / "data" / "raw" / "ml-latest-small")
ratings, movies, tags = data.ratings, data.movies, data.tags
ratings["rated_at"] = pd.to_datetime(ratings["timestamp"], unit="s")
tags["tagged_at"] = pd.to_datetime(tags["timestamp"], unit="s")
print("rows read:    ", data.report.rows_read)
print("rows returned:", data.report.rows_returned)
print("rows dropped: ", data.report.rows_dropped)
print("observations: ", data.report.observations)

rows read:     {'movies': 9742, 'ratings': 100836, 'tags': 3683}
rows returned: {'movies': 9742, 'ratings': 100836, 'tags': 3683}
rows dropped:  {'movies': 0, 'ratings': 0, 'tags': 0}
observations:  {'movies_without_year': 13, 'movies_without_genres': 34, 'tags_empty_text': 0, 'tags_duplicate_rows': 0, 'tag_authors_without_ratings': 0, 'users_in_ratings': 610, 'movies_without_ratings': 18}


## 1. Size and sparsity

In [2]:
n_users, n_movies_rated = ratings["user_id"].nunique(), ratings["movie_id"].nunique()
density = len(ratings) / (n_users * len(movies))
print(
    f"users: {n_users}, movies in catalogue: {len(movies)}, movies with >=1 rating: {n_movies_rated}"
)
print(f"ratings: {len(ratings)}")
print(
    f"density (share of user-movie pairs rated): {density:.4%}  -> {1 - density:.2%} of the matrix is empty"
)
print(f"movies with no ratings: {len(movies) - n_movies_rated}")

users: 610, movies in catalogue: 9742, movies with >=1 rating: 9724
ratings: 100836
density (share of user-movie pairs rated): 1.6968%  -> 98.30% of the matrix is empty
movies with no ratings: 18


## 2. Rating values and the 'relevant item' threshold

In [3]:
dist = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
dist["share"] = dist["count"] / dist["count"].sum()
fig = px.bar(dist, x="rating", y="count", title="How often each rating value is used")
fig.update_xaxes(type="category")
fig.show()
print(
    f"mean rating: {ratings['rating'].mean():.3f}, median: {ratings['rating'].median()}"
)
rows = []
for threshold in (3.5, 4.0, 4.5):
    relevant = ratings["rating"] >= threshold
    per_user = (
        ratings[relevant].groupby("user_id").size().reindex(range(1, 611), fill_value=0)
    )
    rows.append(
        {
            "threshold": f">= {threshold}",
            "share_of_ratings": relevant.mean(),
            "users_with_>=1": int((per_user >= 1).sum()),
            "users_with_>=5": int((per_user >= 5).sum()),
            "users_with_>=10": int((per_user >= 10).sum()),
            "median_per_user": float(per_user.median()),
        }
    )
pd.DataFrame(rows).round(3)

mean rating: 3.502, median: 3.5


,threshold,share_of_ratings,users_with_>=1,users_with_>=5,users_with_>=10,median_per_user
0,>= 3.5,0.612,609,604,589,46.0
1,>= 4.0,0.482,609,603,579,40.0
2,>= 4.5,0.216,598,541,433,18.5


## 3. Activity: the long tail of users and movies

In [4]:
per_user = ratings.groupby("user_id").size()
per_movie = ratings.groupby("movie_id").size()
q = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0]
print(
    "ratings per user (quantiles):\n",
    per_user.quantile(q).round(0).astype(int).to_string(),
)
print(
    "\nratings per movie (quantiles):\n",
    per_movie.quantile(q).round(0).astype(int).to_string(),
)
print(
    f"\nmovies with fewer than 5 ratings: {(per_movie < 5).mean():.1%} of rated movies"
)
print(f"movies with exactly 1 rating: {(per_movie == 1).mean():.1%} of rated movies")
top_share = per_movie.sort_values(ascending=False).cumsum() / per_movie.sum()
for k in (10, 100, 1000):
    print(f"top {k} movies hold {top_share.iloc[k - 1]:.1%} of all ratings")
curve = pd.DataFrame(
    {"rank": np.arange(1, len(top_share) + 1), "cumulative_share": top_share.to_numpy()}
)
fig = px.line(
    curve.iloc[::20],
    x="rank",
    y="cumulative_share",
    title="Popularity concentration: share of all ratings held by the top-N movies",
)
fig.show()

ratings per user (quantiles):
 0.00      20
0.10      24
0.25      35
0.50      70
0.75     168
0.90     400
0.99    1256
1.00    2698

ratings per movie (quantiles):
 0.00      1
0.10      1
0.25      1
0.50      3
0.75      9
0.90     27
0.99    115
1.00    329

movies with fewer than 5 ratings: 62.5% of rated movies
movies with exactly 1 rating: 35.4% of rated movies
top 10 movies hold 2.7% of all ratings
top 100 movies hold 16.1% of all ratings
top 1000 movies hold 60.7% of all ratings


## 4. Genres and release years

In [5]:
hot = genres_multi_hot(movies)
genre_counts = (
    hot.sum()
    .sort_values(ascending=False)
    .rename("movies")
    .reset_index()
    .rename(columns={"index": "genre"})
)
fig = px.bar(
    genre_counts,
    x="movies",
    y="genre",
    orientation="h",
    title="Movies per genre (a movie can have several)",
)
fig.update_yaxes(autorange="reversed")
fig.show()
print(
    f"movies with no genre: {(hot.sum(axis=1) == 0).sum()}; "
    f"average genres per movie: {hot.sum(axis=1).mean():.2f}"
)
decade = (
    (movies["year"].dropna() // 10 * 10)
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("decade")
    .reset_index(name="movies")
)
px.bar(decade, x="decade", y="movies", title="Movies by release decade").show()
rated_genre = ratings.merge(movies[["movie_id"]], on="movie_id")
genre_ratings = (hot.reindex(rated_genre["movie_id"]).sum()).sort_values(
    ascending=False
)
print("\nshare of ratings touching each of the top 5 genres:")
print((genre_ratings.head(5) / len(ratings)).round(3).to_string())

movies with no genre: 34; average genres per movie: 2.26



share of ratings touching each of the top 5 genres:
Drama        0.416
Comedy       0.387
Action       0.304
Thriller     0.262
Adventure    0.240


## 5. Tags (needed by the content-based model)

In [6]:
tags["tag_norm"] = tags["tag"].str.strip().str.lower()
tagged_movies = tags["movie_id"].nunique()
rated_ids = set(ratings["movie_id"])
tagged_and_rated = len(set(tags["movie_id"]) & rated_ids)
print(
    f"tag applications: {len(tags)}, distinct tag texts (lower-cased): {tags['tag_norm'].nunique()}"
)
print(
    f"users who applied any tag: {tags['user_id'].nunique()} of {ratings['user_id'].nunique()}"
)
print(
    f"movies with >=1 tag: {tagged_movies} = {tagged_movies / len(movies):.1%} of the catalogue"
)
print(
    f"share of all ratings that are on a tagged movie: {ratings['movie_id'].isin(set(tags['movie_id'])).mean():.1%}"
)
print(
    "tags per tagged movie (quantiles):",
    tags.groupby("movie_id").size().quantile([0.5, 0.9, 1.0]).astype(int).to_dict(),
)
top_tags = (
    tags["tag_norm"]
    .value_counts()
    .head(15)
    .rename_axis("tag")
    .reset_index(name="applications")
)
fig = px.bar(
    top_tags,
    x="applications",
    y="tag",
    orientation="h",
    title="15 most applied tags (counts only)",
)
fig.update_yaxes(autorange="reversed")
fig.show()
tag_year = (
    tags["tagged_at"]
    .dt.year.value_counts()
    .sort_index()
    .rename_axis("year")
    .reset_index(name="tags")
)
px.bar(tag_year, x="year", y="tags", title="Tag applications by year").show()

tag applications: 3683, distinct tag texts (lower-cased): 1475


users who applied any tag: 58 of 610
movies with >=1 tag: 1572 = 16.1% of the catalogue
share of all ratings that are on a tagged movie: 47.9%
tags per tagged movie (quantiles): {0.5: 1, 0.9: 5, 1.0: 181}


## 6. Time: when were ratings made?

In [7]:
year = (
    ratings["rated_at"]
    .dt.year.value_counts()
    .sort_index()
    .rename_axis("year")
    .reset_index(name="ratings")
)
px.bar(year, x="year", y="ratings", title="Ratings per calendar year").show()
print("timestamp quantiles (date of the Nth percentile of all ratings):")
qs = [0.0, 0.1, 0.25, 0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0]
print(
    pd.to_datetime(
        ratings["timestamp"].quantile(qs).astype("int64"), unit="s"
    ).dt.date.to_string()
)
span = ratings.groupby("user_id")["rated_at"].agg(["min", "max"])
span["days"] = (span["max"] - span["min"]).dt.total_seconds() / 86400
print("\nper-user span between first and last rating (days), quantiles:")
print(span["days"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())
for days in (1, 7, 30, 365):
    print(
        f"users whose ALL ratings fall within {days:>3} day(s): {(span['days'] <= days).mean():.1%}"
    )
users_per_year = (
    ratings.groupby(ratings["rated_at"].dt.year)["user_id"]
    .nunique()
    .rename_axis("year")
    .reset_index(name="active_users")
)
px.bar(
    users_per_year,
    x="year",
    y="active_users",
    title="Users who rated at least once, per year",
).show()
month = ratings["rated_at"].dt.to_period("M").value_counts()
print(
    f"\nbusiest single month holds {month.max() / len(ratings):.1%} of all ratings; top 5 months hold {month.head(5).sum() / len(ratings):.1%}"
)

timestamp quantiles (date of the Nth percentile of all ratings):
0.00    1996-03-29
0.10    1999-11-18
0.25    2002-04-18
0.50    2007-08-02
0.70    2013-08-18
0.80    2016-03-22
0.85    2016-11-19
0.90    2017-06-21
0.95    2018-02-20
1.00    2018-09-24

per-user span between first and last rating (days), quantiles:
0.10      0.0
0.25      0.0
0.50      0.1
0.75     38.8
0.90    616.6
users whose ALL ratings fall within   1 day(s): 59.3%
users whose ALL ratings fall within   7 day(s): 67.5%
users whose ALL ratings fall within  30 day(s): 73.3%
users whose ALL ratings fall within 365 day(s): 85.9%



busiest single month holds 2.4% of all ratings; top 5 months hold 10.1%


## 7. Preview: could a shared chronological split work? (illustration only)

A *shared chronological split* uses two global dates: ratings before the first date are
training data, ratings between the dates are validation, and ratings after the second are test.
**These cutoffs are NOT chosen here.** Step 5 decides them with the owner. The tables below use
three example percentile pairs only to show how the numbers move, so Step 5 starts with evidence.

Definitions used in the preview: a user is *evaluated* in a period if they have at least `m`
ratings before that period starts and at least one relevant rating (>= 4) inside it. A user
with at least one rating in the period but fewer than `m` earlier ones is *cold-start*.
`held-out items known` is the share of relevant held-out ratings whose movie already had at
least one rating before the period.

In [8]:
def preview(lo_q: float, hi_q: float, m: int) -> list[dict[str, object]]:
    t_val = ratings["timestamp"].quantile(lo_q)
    t_test = ratings["timestamp"].quantile(hi_q)
    out = []
    for name, start, end in (("validation", t_val, t_test), ("test", t_test, np.inf)):
        before = ratings[ratings["timestamp"] < start]
        inside = ratings[(ratings["timestamp"] >= start) & (ratings["timestamp"] < end)]
        prior = before.groupby("user_id").size()
        relevant = inside[inside["rating"] >= 4.0]
        users_rel = relevant["user_id"].unique()
        evaluated = [u for u in users_rel if prior.get(u, 0) >= m]
        in_period = inside["user_id"].unique()
        cold = [u for u in in_period if prior.get(u, 0) < m]
        known = (
            relevant["movie_id"].isin(set(before["movie_id"])).mean()
            if len(relevant)
            else float("nan")
        )
        months = inside["rated_at"].dt.to_period("M").value_counts()
        out.append(
            {
                "cutoffs": f"{int(lo_q * 100)}/{int(hi_q * 100)}",
                "period": name,
                "m": m,
                "ratings": len(inside),
                "relevant": len(relevant),
                "users_with_ratings": len(in_period),
                "evaluated_users": len(evaluated),
                "relevant_of_evaluated": int(relevant["user_id"].isin(evaluated).sum()),
                "cold_start_users": len(cold),
                "held_out_items_known": round(float(known), 3),
                "top_month_share": round(
                    float(months.iloc[0] / max(len(inside), 1)), 3
                ),
            }
        )
    return out


rows = []
for lo, hi in ((0.7, 0.85), (0.8, 0.9), (0.9, 0.95)):
    for m in (5, 10, 20):
        rows += preview(lo, hi, m)
table = pd.DataFrame(rows)
table

,cutoffs,period,m,ratings,relevant,users_with_ratings,evaluated_users,relevant_of_evaluated,cold_start_users,held_out_items_known,top_month_share
0,70/85,validation,5,15125,7816,106,14,660,92,0.833,0.122
1,70/85,test,5,15126,6402,96,30,1520,66,0.850,0.158
2,70/85,validation,10,15125,7816,106,14,660,92,0.833,0.122
3,70/85,test,10,15126,6402,96,30,1520,66,0.850,0.158
4,70/85,validation,20,15125,7816,106,13,609,93,0.833,0.122
5,70/85,test,20,15126,6402,96,29,1470,67,0.850,0.158
6,80/90,validation,5,10084,5198,62,21,411,41,0.925,0.238
7,80/90,test,5,10084,4128,74,25,1025,47,0.842,0.184
8,80/90,validation,10,10084,5198,62,21,411,41,0.925,0.238
9,80/90,test,10,10084,4128,74,25,1025,47,0.842,0.184


In [9]:
# How many users have ratings in several periods at the 80/90 example, ignoring m?
t_val, t_test = ratings["timestamp"].quantile([0.8, 0.9])
period = np.select(
    [ratings["timestamp"] < t_val, ratings["timestamp"] < t_test],
    ["train", "val"],
    "test",
)
presence = pd.crosstab(ratings["user_id"], period).gt(0)
print(
    "users with ratings in: train",
    int(presence["train"].sum()),
    "| val",
    int(presence["val"].sum()),
    "| test",
    int(presence["test"].sum()),
)
print(
    "users in train AND val:",
    int((presence["train"] & presence["val"]).sum()),
    "| train AND test:",
    int((presence["train"] & presence["test"]).sum()),
    "| all three:",
    int(presence.all(axis=1).sum()),
)
print(
    "users whose only ratings are in the test period (no history at all):",
    int((presence["test"] & ~presence["train"] & ~presence["val"]).sum()),
)

users with ratings in: train 522 | val 62 | test 74
users in train AND val: 21 | train AND test: 20 | all three: 13
users whose only ratings are in the test period (no history at all): 47


## 8. Findings that feed the plan (numbers come from the cells above)

**Data quality.** Loaders read and returned every row (100,836 ratings, 9,742 movies, 3,683 tags;
0 dropped; 610 users). Observations: 13 movies without a year in the title, 34 without genres,
18 movies with no ratings, no empty or duplicate tags.

**Rating threshold (owner decision still to confirm).** Ratings run 0.5 to 5.0 in half steps, mean
3.50. A threshold of `>= 4.0` makes 48.2% of ratings "relevant" and leaves 603 of 610 users with at
least 5 relevant ratings. `>= 3.5` would call 61.2% relevant (too lenient); `>= 4.5` only 21.6% and
69 users would have fewer than 5. Recommendation: `>= 4.0`.

**Activity and sparsity.** 98.3% of the user-movie matrix is empty. Every user has at least 20
ratings (the dataset's own rule); the median user has 70, the top 1% over 1,256. 62.5% of rated movies
have fewer than 5 ratings and 35.4% have exactly one, so collaborative methods will be weak for most
of the catalogue, while the 1,000 most-rated movies hold 60.7% of all ratings (popularity is a
strong baseline).

**Tags.** 3,683 tag applications, 1,475 distinct lower-cased texts, but only 58 of 610 users ever
tagged, only 1,572 movies (16.1%) have a tag, and the median tagged movie has one tag. Tags cover
47.9% of ratings because tagged movies are popular. Conclusion: tags can add a little to the
content model for popular movies and nothing for most movies; genres and title words must carry
the content model.

**Time and the shared chronological split.** Ratings span 1996-03 to 2018-09, but 59.3% of users made
*all* their ratings within one day and 85.9% within a year. Consequently, at the example cutoffs
(percentiles 70/85, 80/90 and 90/95) only 43 to 106 users have any validation ratings and 45 to 96
any test ratings, and the number of **evaluated** users (enough earlier history *and* a relevant
rating in the period) is only 13 to 30 per period, far below the plan's proposed minimum of 100.
The minimum-history size `m` (5, 10 or 20) barely matters, because a user who rates in two
different periods usually has plenty of history. Relevant held-out ratings from evaluated users
range from 347 to 1,520 per period; cold-start users (some activity, little earlier history) number
22 to 93, and 47 users at the 80/90 example have their only ratings in the test period. Between
82% and 93% of held-out relevant movies already had a rating before the period. Test periods
are time-concentrated: the single busiest month holds 12% to 37% of a period's ratings.

**What this means (for the owner and Step 5).** On this dataset a shared chronological split does
not meet the plan's feasibility criteria (about 13 to 30 evaluated users, against a proposed 100).
The data would still support a *comparison of methods*, but with very wide uncertainty.
The plan's fallback (a per-user holdout, reported with a narrowed claim) or relaxed minimums are the
options Step 5 has to weigh. No cutoffs were chosen in this notebook.